In [1]:
#It is recommended to increase the size of your Jupyter kernel when launching it using the command 

#jupyter notebook --ServerApp.max_buffer_size=1036870912

# General methodology:

-We create in python the list of stair centers.

-We use a $C$ implementation of Masked Dilithium to generate a large number of signatures.
 
-We filter the signatures for which $ Az-ct = w - cs_2 \in $ stair_centers, and store the corresponding $w$, $z$, and $c$ values.

-We use ChipWhisperer on the filtered $w$ values: We only run the decompose function and obtain a vector of guesses $\tilde{w} $.

-We solve the system using LSM or another method. 

Precisions on the system:

The shares of $w$ are given using the masked $C$ implementation by Andersson.

Then the template is run on theses values, it gives a candidate $\tilde{w}$.

For now we recompute $Az-ct$ using $w -cs_2$ (even if we should not), then the noisy equation is given by:

$$ cs2 = w - (w-cs_2) = \tilde{w} + \epsilon - (Az-ct).$$

In [1]:
!pwd

/media/sf_vm_shared/Dev/ML-DSA/Security/Decompose/artifacts/artifacts/clean_template


In [2]:
# Loading auxiliary functions 
probable_path_to_helpers_functions = !find ../Common_functions -name "Helpers.py"
# If the Helper.py file is not found and you don't need it, comment this cell
# If the Helper.py file is not found and you need it, something went wrong ...
print(f"Probable path to Helpers functions:")
print(f">>> {probable_path_to_helpers_functions}")
probable_path_to_helpers_functions = probable_path_to_helpers_functions[0]
%run -i $probable_path_to_helpers_functions

Probable path to Helpers functions:
>>> ['../Common_functions/Helpers.py']


In [4]:
MODE = 2
# K = 3
# K = 5

In [5]:
# Loading ml-dsa parameters according to the chosen security level K
%run -i ../Common_functions/MLDSA_parameters.py {MODE}
# Loading auxiliary functions
%run -i ../Common_functions/MLDSA_functions.py
# Loading auxiliary functions
%run -i ../Common_functions/Additional_functions.py

In [6]:
# Importing useful libraries
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from tqdm.notebook import trange
import copy

import scared
import estraces

from collections import Counter

from scalib.preprocessing import Quantizer
from scalib.metrics import Ttest, SNR

import struct 

from scipy.signal import correlate
from scipy.ndimage import shift

import numpy as np

In [7]:
# Setting default size of figures in Matplotlib
plt.rcParams["figure.figsize"] = (13,3) 

# Set the maximum display width for NumPy arrays
np.set_printoptions(linewidth=sys.maxsize)

# Adjusting the display of tables
np.set_printoptions(threshold=sys.maxsize)

In [8]:
# Matplotlib constants 
span_color = "#FFC069"
color0 = "darkblue"
color1 = "coral"

### Unpacking the shares

In [9]:
nb_key = 9

In [10]:
temp = np.load("./stats/Keys/Key"+str(nb_key)+"/w_shares.npz")
w_share0 = temp['w_share0']
w_share1 = temp['w_share1']

temp2 = np.load("./stats/Keys/Key"+str(nb_key)+"/w_cs2_shares.npz")
wmcs2_share0 = temp2['w_cs2_share0']
wmcs2_share1 = temp2['w_cs2_share1']
wmcs2 = [int((wmcs2_share0[_] + wmcs2_share1[_])%Q) for _ in range(len(wmcs2_share0))]
w = [int((w_share0[_] + w_share1[_])%Q) for _ in range(len(w_share0))]

### Running template on the shares

In [11]:
dataset = f'dataset_synchro.ets'
ths_synchro = estraces.read_ths_from_ets_file(dataset)
ths_synchro

Trace Header Set:
Name.............: ETS Format THS
Reader...........: ETS format reader of file dataset_synchro.ets with 22623 traces.
w................: uint8
w_share0.........: uint8
w_share1.........: uint8
w_z2.............: uint8
w_z3.............: uint8

In [12]:
def hw(string):
    return string.count("1")

v__hw = np.vectorize(hw)

def compute_hw_recombined_z2(w, w_share0, w_share1):
    return v__hw(np.vectorize(np.binary_repr)(-44*np.uint32(w.view(dtype='<u4')).view('int32') +(Q-1)//2, width=32))

def compute_hw_z2(w, w_share0, w_share1, w_z2, w_z3):
    return v__hw(np.vectorize(np.binary_repr)(w_z2.view(dtype='<u4'), width=32))

In [13]:
S = scared.reverse_selection_function(function = compute_hw_z2)

In [14]:
ReverseANOVA = scared.ANOVAReverse(selection_function=S,
                                 model = scared.Value(),
                                )
container = scared.Container(ths_synchro)
ReverseANOVA.run(container)
pois = np.sort(np.argsort(ReverseANOVA.results[0])[-2:])
pois

array([292, 362])

In [15]:
# Base Scope for ChipWhisperer Lite 
SCOPETYPE = 'OPENADC'

# ChipWhisperer Lite used Cortex-M4 
PLATFORM = 'CWLITEARM'

# Project Targeted
CRYPTO_TARGET ='DECOMPOSE2'

# SimpleSerial version used
SS_VER = 'SS_VER_1_1'

In [16]:
# Detecting where is the simpleserial find, if any
probable_path_to_chipwhisperer_setup_notebook = !find /home/paco/Bureau/Formation_ChipWhisperer/chipwhisperer/jupyter/Setup_Scripts/ -name "Setup_Generic.ipynb"

print(f"Probable path to ChipWhisperer jupyter setup script:")
print(f">>> {probable_path_to_chipwhisperer_setup_notebook}")

Probable path to ChipWhisperer jupyter setup script:
>>> ['/home/paco/Bureau/Formation_ChipWhisperer/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb']


In [17]:
# # Change the path to match the location of your file Setup_Generic.ipynb
# path_to_chipwhisperer_setup_notebook = ...
path_to_chipwhisperer_setup_notebook = probable_path_to_chipwhisperer_setup_notebook[0]

In [18]:
%run $path_to_chipwhisperer_setup_notebook

(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.64.0) is outdated - latest is 0.65.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 70468670                  to 103466861                
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 96000000                  to 29538471                 
scope.clock.adc_rate                     changed from 96000000.0                to 29538471.0        

In [19]:
# Maximum number of samples allowed in [0, 24400]
scope.adc.samples = 4400

In [ ]:
# Detecting where is the simpleserial location, if any
probable_path_to_simpleserial = !find ~ -name simpleserial-{CRYPTO_TARGET.lower()}

print(f"Probable path to communication protocol with the {CRYPTO_TARGET} code: ")
print(f">>> {probable_path_to_simpleserial}")

In [21]:
# # Change the path to match the location of the simpleserial code 
# path_to_simpleserial = ... 
path_to_simpleserial = probable_path_to_simpleserial[-1]

In [22]:
%%bash -s "$path_to_simpleserial" "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER" 
cd $1
make -s clean PLATFORM=$2 CRYPTO_TARGET=$3 SS_VER=$4 
make PLATFORM=$2 CRYPTO_TARGET=$3 SS_VER=$4 

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
.
Cleaning project:
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:12.2.rel1-1) 12.2.1 20221205
Copyright (C) 2022 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Compiling:
-en     simpleserial-decompose2.c ...


simpleserial-decompose2.c: In function 'setup2':
simpleserial-decompose2.c:473:41: warning: passing argument 3 of 'simpleserial_put' from incompatible pointer type [-Wincompatible-pointer-types]
  473 |     simpleserial_put('r', 4*N_SHARES*2, &rz2_rz3);
      |                                         ^~~~~~~~
      |                                         |
      |                                         uint32_t (*)[4] {aka long unsigned int (*)[4]}
In file included from simpleserial-decompose2.c:5:
../simpleserial/simpleserial.h:63:54: note: expected 'uint8_t *' {aka 'unsigned char *'} but argument is of type 'uint32_t (*)[4]' {aka 'long unsigned int (*)[4]'}
   63 | void simpleserial_put(char c, uint8_t size, uint8_t* output);
      |                                             ~~~~~~~~~^~~~~~


-e Done!
.
Compiling:
-en     ../simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     ../hal/hal.c ...
-e Done!
.
Compiling:
-en     ../hal//stm32f3/stm32f3_hal.c ...
-e Done!
.
Compiling:
-en     ../hal//stm32f3/stm32f3_hal_lowlevel.c ...
-e Done!
.
Compiling:
-en     ../hal//stm32f3/stm32f3_sysmem.c ...
-e Done!
.
Assembling: ../hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I../simpleserial/ -I../hal/ -I../hal/ -I../hal//stm32f3 -I../hal//stm32f3/CMSIS -I../hal//stm32f3/CMSIS/core -I../hal//stm32f3/CMSIS/device -I../hal//stm32f4/Legacy -I../simpleserial/ -I../crypto/ ../hal//stm32f3/stm32f3_startup.S -o objdir-CWLITEARM/stm32f3_startup.o
.
LINKING:
-en     simpleserial-decompose2-CWLITEARM.elf ...
Memory region         Used Size  Region Size  %age Used
             RAM:        1704 B        4

In [ ]:
# Detecting where is the simpleserial find, if any
probable_path_to_executable = !find ~/ -name simpleserial-{CRYPTO_TARGET.lower()}-{PLATFORM}.hex

print(f"Probable path to communication protocol with the {CRYPTO_TARGET} code: ")
print(f">>> {probable_path_to_executable}")

In [24]:
# # Change the path to match the executable code 
# path_to_executable = ...
path_to_executable = probable_path_to_executable[-1]

In [25]:
cw.program_target(scope, prog, path_to_executable)

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 8567 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 8567 bytes


In [26]:
vhex = np.vectorize(hex)

def cw_to_int(hexstr):
    beh = bytes.fromhex(hexstr)
    leh = struct.unpack("<I", beh)[0]
    return leh

def cw_to_int2(byte_list):
    leb = bytes(reversed(byte_list))
    num = int.from_bytes(leb, "big", signed=True)
    return num

def int_to_cw2(num):
    return [int(byy) for byy in reversed((num).to_bytes(4, "big", signed=True))]

def int_to_cw(num, b_ = 4):
    leb = num.to_bytes(b_, "big", signed=True)
    beh = "".join(format(byte, "02x") for byte in reversed(leb))
    return beh

def from_intarray_to_kybermsg(arrray):
    return "".join([f"{byyte:08x}"[::-1] for byyte in arrray])

def HW(n):
    return bin(n).count('1')

def compressed(val):
    msg_int = cw_to_int2(val)
    if msg_int > -Q//4:
        return [byy for byy in reversed((0).to_bytes(4, "big", signed=True))]
    return [byy for byy in reversed((1).to_bytes(4, "big", signed=True))]

def bytes_to_hex_str(bytess):
    return ''.join('{:02x}'.format(x) for x in bytess)

def int_array_to_hex_str(int_array):
    return "".join([int_to_cw(int(int_array[i])) for i in range(len(int_array))])

def hex_str_to_int_array(hex_str, hex_str_len = 8):
    int_aray = []
    for offset in range(0, len(hex_str), hex_str_len):
        int_aray.append(cw_to_int(hex_str[offset:offset + hex_str_len]))
    return int_aray

def hw(string):
    return string.count("1")
v__hw = np.vectorize(hw)

In [27]:
def trace_worker(scope, target, n_samples, w_shares):
    t = np.zeros((n_samples))
    scope.arm()
    target.simpleserial_write('d', bytearray(w_shares))
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')
    else:
        t = scope.get_last_trace()[:n_samples]
    test_output = target.simpleserial_read("r", 2*2*4)
    z2_share0, z2_share1 =  cw_to_int2(test_output[:4]), cw_to_int2(test_output[4:8])
    z3_share0, z3_share1 =  cw_to_int2(test_output[8:12]), cw_to_int2(test_output[12:16])

    z2, z3 = z2_share0 ^ z2_share1, z3_share0 ^ z3_share1
    return t, z2, z3

In [28]:
Y_z2 = np.load("../dump_z2.npy").tolist()
Y_z3 = np.load("../dump_z3.npy").tolist()

Y_max = []
for _ in range(Q-1):
    Y_max.append(max(Y_z2[_],Y_z3[_]))

In [29]:
```python
def split_into_two_groups_min_gap_4(numbers):
    """
    Split a list of numbers into two groups of consecutive numbers,
    such that the distance between the end of the first group and the
    start of the second group is at least 4.

    Returns:
        (group1, group2) if possible,
        ([], []) otherwise.
    """
    if not numbers:
        return [], []

    sorted_unique = sorted(set(numbers))

    groups = []
    current_group = [sorted_unique[0]]

    for n in sorted_unique[1:]:
        if n == current_group[-1] + 1:
            current_group.append(n)
        else:
            groups.append(current_group)
            current_group = [n]
    groups.append(current_group)

    for i in range(1, len(groups)):
        if groups[i][0] - groups[i - 1][-1] >= 4:
            group1 = [x for grp in groups[:i] for x in grp]
            group2 = [x for grp in groups[i:] for x in grp]
            return group1, group2

    return [], []
```

In [30]:
usfl_pts = list(np.fromfile('./useful_coeffs.bin', dtype=np.int32))

In [31]:
usfl_pts[0:10]

[np.int32(635),
 np.int32(1041),
 np.int32(1380),
 np.int32(1381),
 np.int32(1413),
 np.int32(2158),
 np.int32(2869),
 np.int32(2870),
 np.int32(2901),
 np.int32(2902)]

The values we will store:

- The position of the value w in the list.

- The value w provided by the template.

- Was the interval correct?

- The actual Hamming weight.

- The Hamming weight provided by the template.

In [32]:
print("Remainder: nb_key = ",nb_key)

Remainder: nb_key =  9


In [33]:
temp = np.load("./stats/Keys/Key"+str(nb_key)+"/RES.npz")

In [34]:
ind_w = list(temp["ind_w"])
val_g = list(temp["val_g"])
int_ok = list(temp["int_ok"])
hw_z2 = list(temp["hw_z2"])
hw_g = list(temp["hw_g"])

ind_w = []
val_g = []
int_ok = []
hw_z2 = []
hw_g =[]

In [35]:
len(ind_w)

1879

In [36]:
ind_w[-1]

np.int64(3997)

In [37]:
bloc = 2

In [38]:
bloc*2000

4000

In [39]:
n_samples = 20000
scope.adc.samples = n_samples
scope.adc.offset = 0
scope.clock.adc_src = 'clkgen_x1'
target = cw.target(scope, cw.targets.SimpleSerial, flush_on_err=False)
nb_trace_0, nb_synchro, template_positive = 0, 0, []
for i in trange(bloc*2000, bloc*2000+2000, desc = "Testing"): 
    wmcs2_i =wmcs2[i]
    H = [Y_max[wmcs2_i+_] for _ in range(-17,18)]
    H_i = split_into_two_groups_min_gap_4(H)
    big = H_i[1]
    low = H_i[0]
    zero = [0]

    w_shares = np.array([w_share0[i], w_share1[i]], dtype="uint32")
    w_intermediate = int((w_share0[i] + w_share1[i]) % Q)
    trace, w_z2, w_z3 = trace_worker(scope, target, n_samples, w_shares)

    if w_z2 != 0:
        
        shift_matching = None
        max_cc = 0
        for j in range(11242-100, 11242+100):
            cc = np.corrcoef(list(ths_synchro.samples[0]), trace[j : j + ths_synchro.samples[0].shape[0]])[0, 1]
            if cc >= max_cc:
                max_cc = cc
                shift_matching = j
    
        if shift_matching is not None:
            nb_synchro+=1
            new_trace_reshaped = trace[shift_matching:shift_matching + ths_synchro.samples[0].shape[0]] 
         
            TAttacks = scared.TemplateAttack(container_building = scared.Container(ths_synchro, frame = pois),
                                            reverse_selection_function = scared.reverse_selection_function(function = compute_hw_z2),
                                            model = scared.Value(),
                                            convergence_step = 1,
                                            partitions =  np.array([0]+low+big))
            TAttacks.build()
            ths_matching = estraces.read_ths_from_ram(np.array([new_trace_reshaped]),
                                                    w =  np.array([np.array(int_to_cw2(w_intermediate), dtype = np.uint8)]),
                                                    w_share0 = np.array([np.array(int_to_cw2(int(w_shares[0])), dtype = np.uint8)]),
                                                    w_share1 = np.array([np.array(int_to_cw2(int(w_shares[1])), dtype = np.uint8)]),
                                                    w_z2 = np.array([np.array(int_to_cw2(int(w_z2)), dtype = np.uint8)]),
                                                    w_z3 = np.array([np.array(int_to_cw2(int(w_z3)), dtype = np.uint8)]))
            TAttacks.run (scared.Container(ths_matching, frame = pois))
            K = TAttacks.scores.argmax(0).squeeze()
            g_HW = TAttacks.partitions[K]
            secret_key = compute_hw_z2(ths_matching.w[0], ths_matching.w_share0[0], ths_matching.w_share1[0], ths_matching.w_z2[0], ths_matching.w_z3[0])[0]
            #print("guessed, secret hw",g_HW, secret_key)
            ths_matching.close()
            
            nb_trace_0 += 1 
            H = [[wmcs2_i+_ for _ in range(-17,18) if Y_max[wmcs2_i+_] in low], [wmcs2_i+_ for _ in range(-17,18) if Y_max[wmcs2_i+_] in big]]
            H = H[g_HW in big] #low if g_HW in low, big if not
            if len(H)<17:
                ind_w.append(i)
                val_g.append(H[len(H)//2])
                int_ok.append( (g_HW in big and secret_key in big) or (g_HW in low and secret_key in low) )
                hw_z2.append(secret_key)
                hw_g.append(g_HW)
bloc+=1
np.savez("./stats/Keys/Key"+str(nb_key)+"/RES.npz", ind_w = ind_w, val_g = val_g, int_ok = int_ok, hw_z2 = hw_z2, hw_g = hw_g)

Testing:   0%|          | 0/2000 [00:00<?, ?it/s]

In [41]:
disconnect_cw()

ChipWhisperer disconnected, Goodbye! 😢
